# Quantalytics — small, safe schema-drift demonstration

This mini-lab is for the **viva**. It makes three fake IBM JSON strings in memory. It does **not** edit Bronze or Silver tables. It shows a valid record, a changed numeric type and an unexpected new field.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, MapType

# Three invented rows. The second changes a number to text.
# The third adds a column that is not in our expected schema.
raw_records = [
    ('{"Qubit":"7","T1 (us)":"100.5","Backend_ID":"ibm_fez"}',),
    ('{"Qubit":"7","T1 (us)":"hello","Backend_ID":"ibm_fez"}',),
    ('{"Qubit":"7","T1 (us)":"100.5","Backend_ID":"ibm_fez","new_column":"oops"}',),
]
raw_schema = StructType([StructField("raw_payload", StringType(), False)])
raw = spark.createDataFrame(raw_records, raw_schema)

expected = ["Qubit", "T1 (us)", "Backend_ID"]
json_schema = StructType([StructField(name, StringType(), True) for name in expected])

In [0]:
checked = (raw
    .withColumn("p", F.from_json("raw_payload", json_schema))
    .withColumn("keys", F.map_keys(F.from_json("raw_payload", MapType(StringType(), StringType()))))
    .withColumn("extra_keys", F.array_except("keys", F.array(*[F.lit(c) for c in expected])))
    .withColumn("t1", F.expr("try_cast(p.`T1 (us)` AS DOUBLE)"))
    .withColumn("result", F.when(F.size("extra_keys") > 0, "SCHEMA DRIFT: QUARANTINE")
              .when(F.col("t1").isNull(), "TYPE ERROR: QUARANTINE")
              .otherwise("VALID: ACCEPT")))

display(checked.select("raw_payload", "extra_keys", "t1", "result"))

raw_payload,extra_keys,t1,result
"{""Qubit"":""7"",""T1 (us)"":""100.5"",""Backend_ID"":""ibm_fez""}",List(),100.5,VALID: ACCEPT
"{""Qubit"":""7"",""T1 (us)"":""hello"",""Backend_ID"":""ibm_fez""}",List(),null,TYPE ERROR: QUARANTINE
"{""Qubit"":""7"",""T1 (us)"":""100.5"",""Backend_ID"":""ibm_fez"",""new_column"":""oops""}",List(new_column),100.5,SCHEMA DRIFT: QUARANTINE


Expected: 1 valid row, 1 bad numeric value and 1 new-column drift. Explain: **we use an explicit expected schema; identify changes instead of silently trusting new fields; and quarantine suspicious records in our real Silver notebook.** This demo is intentionally not a production table update.